# 面试问题：Speculative Decoding 如何保证分布正确并减少大模型调用？

可以直接复述的回答是：小 draft model 先连续提议 k 个 token，大 target model 一次验证整个块。第 i 个提议以 `min(1, p_i(x)/q_i(x))` 接受；首次拒绝时从归一化残差 `max(p-q,0)` 采样修正 token，并丢弃后续草稿。若整块都接受，再从 target 的下一位置采一个 bonus token。这个接受/残差机制保证边缘分布仍等于 target；“直接相信 draft”虽然更快但会产生分布偏差。收益取决于接受率、块长和 target 批量验证效率。下面用手写 bigram 模型和可复现采样循环验证。

## 真实案例：六类短回复的词级生成

六个 prompt 分别从退款、物流、技术、安全、会员和问候起始 token 开始。词表和转移概率为教学构造的离线小模型，不代表真实 LLM；但 draft/target 概率、接受门槛和 residual sampling 与实际算法同构。

In [1]:
import warnings  # 导入告警控制模块
import torch  # 导入 PyTorch 张量与概率运算
warnings.filterwarnings("ignore")  # 隐藏环境告警保持输出清晰
torch.set_num_threads(1)  # 固定 CPU 单线程执行
torch.manual_seed(1303)  # 固定模型表初始化
vocabulary = ["退款", "物流", "排查", "安全", "完成", "<eos>"]  # 定义六个可读生成 token
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 建立 token 到整数 ID 映射
target_probabilities = torch.tensor([  # 定义 target bigram 条件概率表
    [0.05, 0.05, 0.05, 0.05, 0.65, 0.15],  # 退款后通常生成完成
    [0.05, 0.10, 0.05, 0.05, 0.60, 0.15],  # 物流后通常生成完成
    [0.05, 0.05, 0.20, 0.05, 0.50, 0.15],  # 排查后生成完成或继续排查
    [0.05, 0.05, 0.05, 0.25, 0.45, 0.15],  # 安全后可能继续安全说明
    [0.03, 0.03, 0.03, 0.03, 0.08, 0.80],  # 完成后高概率结束
    [0.00, 0.00, 0.00, 0.00, 0.00, 1.00],  # 结束标记保持吸收
], dtype=torch.float32)  # 结束 target 转移表
draft_probabilities = torch.tensor([  # 定义近似但并不完全一致的 draft 表
    [0.08, 0.06, 0.06, 0.05, 0.55, 0.20],  # 退款行略低估完成
    [0.06, 0.14, 0.06, 0.05, 0.50, 0.19],  # 物流行偏向重复物流
    [0.06, 0.06, 0.30, 0.05, 0.38, 0.15],  # 排查行过度重复排查
    [0.06, 0.05, 0.05, 0.35, 0.34, 0.15],  # 安全行过度重复安全
    [0.05, 0.04, 0.04, 0.04, 0.12, 0.71],  # 完成行低估结束
    [0.00, 0.00, 0.00, 0.00, 0.00, 1.00],  # 结束标记保持吸收
], dtype=torch.float32)  # 结束 draft 转移表
prompts = [  # 定义六个业务起始 prompt
    ("SP-01", "退款进度", "退款"),  # 退款回复生成
    ("SP-02", "包裹状态", "物流"),  # 物流回复生成
    ("SP-03", "设备故障", "排查"),  # 技术回复生成
    ("SP-04", "账户异常", "安全"),  # 安全回复生成
    ("SP-05", "会员已开通", "完成"),  # 会员确认生成
    ("SP-06", "问题已解决", "完成"),  # 结束确认生成
]  # 结束六个 prompt
class BigramLM(torch.nn.Module):  # 定义可批量验证前缀的最小语言模型
    def __init__(self, probabilities):  # 初始化固定转移 logits
        super().__init__()  # 初始化 PyTorch 模块基类
        self.register_buffer("logits", torch.log(probabilities + 1e-12))  # 用 buffer 保存冻结概率的对数值
    def forward(self, previous_tokens):  # 根据前一个 token 返回下一 token logits
        return self.logits[previous_tokens]  # 执行整数索引得到条件分布
target_model = BigramLM(target_probabilities)  # 创建昂贵 target 教学模型
draft_model = BigramLM(draft_probabilities)  # 创建便宜 draft 教学模型
print("输入预览：id | prompt | start_token")  # 输出六个生成请求表头
for prompt in prompts:  # 逐条展示 prompt
    print(prompt)  # 展示请求身份、语义和起始 token
print("退款后的 target/draft：", target_probabilities[0].tolist(), draft_probabilities[0].tolist())  # 展示两个模型真实概率差

输入预览：id | prompt | start_token
('SP-01', '退款进度', '退款')
('SP-02', '包裹状态', '物流')
('SP-03', '设备故障', '排查')
('SP-04', '账户异常', '安全')
('SP-05', '会员已开通', '完成')
('SP-06', '问题已解决', '完成')
退款后的 target/draft： [0.05000000074505806, 0.05000000074505806, 0.05000000074505806, 0.05000000074505806, 0.6499999761581421, 0.15000000596046448] [0.07999999821186066, 0.05999999865889549, 0.05999999865889549, 0.05000000074505806, 0.550000011920929, 0.20000000298023224]


## Baseline / 基线：Target Model 每个 token 调用一次

基线按 target 分布逐 token 采样。生成多少 token 就需要多少次 target forward。

In [2]:
def categorical_sample(probabilities, generator):  # 用显式随机生成器手写分类采样
    draw = float(torch.rand((), generator=generator))  # 生成零到一之间的可复现随机数
    cumulative = 0.0  # 初始化累计概率
    selected = len(probabilities) - 1  # 预设浮点边界下最后 token
    for index, probability in enumerate(probabilities):  # 按词表顺序扫描概率区间
        cumulative += float(probability)  # 更新当前累计上界
        if draw <= cumulative:  # 检查随机数是否落入当前区间
            selected = index  # 选择命中的 token ID
            break  # 结束本次采样
    return selected, draw  # 返回 token 和用于审计的随机数
def target_only_decode(start_id, seed, maximum_tokens=6):  # 实现逐 token target 自回归基线
    generator = torch.Generator().manual_seed(seed)  # 创建请求级随机流
    generated = []  # 收集新生成 token
    target_calls = 0  # 统计昂贵模型 forward 次数
    previous = start_id  # 从 prompt 最后 token 开始
    for step in range(maximum_tokens):  # 限制最大生成长度
        logits = target_model(torch.tensor([previous]))[0]  # 每一步单独调用 target forward
        probability = torch.softmax(logits, dim=0)  # 把 target logits 归一化为概率
        token, draw = categorical_sample(probability, generator)  # 从权威分布采样下一 token
        target_calls += 1  # 累加一次 target 调用
        generated.append(token)  # 保存生成 token
        previous = token  # 更新自回归状态
        if token == token_to_id["<eos>"]:  # 检查结束标记
            break  # 遇到结束 token 停止
    return generated, target_calls  # 返回序列和昂贵调用数
baseline_results = []  # 收集六个 target-only 结果
print("id | target-only tokens | target_calls")  # 输出基线解码表头
for index, prompt in enumerate(prompts):  # 遍历六个业务请求
    tokens, calls = target_only_decode(token_to_id[prompt[2]], 100 + index)  # 用独立种子执行权威采样
    baseline_results.append((tokens, calls))  # 保存 token ID 和调用成本
    print(f"{prompt[0]} | {[vocabulary[token] for token in tokens]} | {calls}")  # 展示真实生成序列

id | target-only tokens | target_calls
SP-01 | ['排查', '完成', '<eos>'] | 3
SP-02 | ['排查', '完成', '安全', '<eos>'] | 4
SP-03 | ['退款', '完成', '物流', '完成', '<eos>'] | 5
SP-04 | ['完成', '<eos>'] | 2
SP-05 | ['<eos>'] | 1
SP-06 | ['<eos>'] | 1


## 核心实现：draft 提议、target 批量验证、接受与 residual 修正

教学 bigram 的 target forward 可以一次接收整个 draft 前驱序列，计为一次批量调用。首次拒绝后立即从 `max(p-q,0)` 归一化分布采样。

In [3]:
def speculative_decode(start_id, seed, block_size=3, maximum_tokens=6, verbose=False):  # 手写推测解码完整循环
    generator = torch.Generator().manual_seed(seed)  # 创建请求级可复现随机流
    generated = []  # 收集最终输出 token
    previous = start_id  # 保存当前已确认前缀最后 token
    target_calls = 0  # 统计 target 批量验证次数
    accepted_count = 0  # 统计被接受 draft token 数
    trace = []  # 保存每个提议的概率与门槛
    while len(generated) < maximum_tokens and previous != token_to_id["<eos>"]:  # 在长度和结束条件内循环块解码
        proposals = []  # 收集当前 draft block token
        proposal_q = []  # 收集每个提议在 draft 下的概率
        draft_previous = previous  # 从已确认前缀开始 draft 自回归
        for draft_step in range(block_size):  # 连续生成最多 k 个草稿 token
            draft_distribution = torch.softmax(draft_model(torch.tensor([draft_previous]))[0], dim=0)  # 调用 draft 得到下一 token 分布
            proposal, draft_draw = categorical_sample(draft_distribution, generator)  # 从 draft 分布采样提议
            proposals.append(proposal)  # 保存提议 token
            proposal_q.append(float(draft_distribution[proposal]))  # 保存 q(x) 用于接受概率
            draft_previous = proposal  # 更新 draft 自回归状态
            if proposal == token_to_id["<eos>"]:  # 草稿提前生成结束标记
                break  # 停止继续提议
        verification_inputs = torch.tensor([previous] + proposals[:-1], dtype=torch.long)  # 构造每个提议对应的 target 前驱 token
        target_logits = target_model(verification_inputs)  # 一次批量 target forward 验证整个草稿块
        target_distributions = torch.softmax(target_logits, dim=1)  # 获取每个提议位置的权威分布
        target_calls += 1  # 整个 block 只计一次昂贵调用
        rejected = False  # 初始化当前块尚未拒绝
        for proposal_index, proposal in enumerate(proposals):  # 按顺序验证每个草稿 token
            p_value = float(target_distributions[proposal_index, proposal])  # 读取 target 对提议 token 的概率
            q_value = proposal_q[proposal_index]  # 读取 draft 提议概率
            threshold = min(1.0, p_value / q_value)  # 计算严格接受概率
            uniform = float(torch.rand((), generator=generator))  # 采样独立接受随机数
            accepted = uniform <= threshold  # 判断当前提议是否通过校验
            trace.append((vocabulary[proposal], p_value, q_value, threshold, uniform, accepted))  # 保存可审计接受轨迹
            if accepted:  # 处理通过 target 校验的 token
                generated.append(proposal)  # 将草稿 token 加入最终输出
                previous = proposal  # 更新已确认前缀
                accepted_count += 1  # 累加接受数量
                if proposal == token_to_id["<eos>"] or len(generated) >= maximum_tokens:  # 检查结束或长度限制
                    break  # 停止当前块验证
            else:  # 处理首次被 target 拒绝的提议
                residual = torch.clamp(target_distributions[proposal_index] - torch.softmax(draft_model(torch.tensor([verification_inputs[proposal_index]]))[0], dim=0), min=0.0)  # 计算非负 target-draft 残差
                residual = residual / residual.sum() if float(residual.sum()) > 0.0 else target_distributions[proposal_index]  # 归一化残差并提供数值回退
                correction, correction_draw = categorical_sample(residual, generator)  # 从修正分布采样替代 token
                generated.append(correction)  # 只提交修正 token 并丢弃后续草稿
                previous = correction  # 更新已确认前缀
                rejected = True  # 标记当前 block 已发生拒绝
                break  # 首次拒绝后停止验证剩余提议
        if previous == token_to_id["<eos>"] or len(generated) >= maximum_tokens:  # 检查当前块是否已经完成生成
            break  # 结束整体解码
        if not rejected and len(generated) < maximum_tokens:  # 整块接受时生成一个 target bonus token
            bonus_distribution = torch.softmax(target_model(torch.tensor([previous]))[0], dim=0)  # 用 target 计算下一位置分布
            bonus, bonus_draw = categorical_sample(bonus_distribution, generator)  # 采样额外权威 token
            target_calls += 1  # 教学实现把 bonus forward 单独计一次
            generated.append(bonus)  # 提交 bonus token
            previous = bonus  # 更新已确认前缀
    if verbose:  # 根据教学开关输出完整提议验证轨迹
        print("token | p | q | min(1,p/q) | u | accepted")  # 输出接受循环表头
        for item in trace:  # 遍历每个被验证提议
            print(f"{item[0]} | {item[1]:.3f} | {item[2]:.3f} | {item[3]:.3f} | {item[4]:.3f} | {item[5]}")  # 展示概率比与随机门槛
    return generated[:maximum_tokens], target_calls, accepted_count, trace  # 返回输出、成本和接受轨迹
sample_tokens, sample_calls, sample_accepted, sample_trace = speculative_decode(token_to_id["排查"], 102, verbose=True)  # 对技术 prompt 展示完整接受过程
print("技术 prompt 输出：", [vocabulary[token] for token in sample_tokens], "target_calls=", sample_calls, "accepted=", sample_accepted)  # 汇总样例解码

token | p | q | min(1,p/q) | u | accepted
退款 | 0.050 | 0.060 | 0.833 | 0.426 | True
完成 | 0.650 | 0.550 | 1.000 | 0.616 | True
物流 | 0.030 | 0.040 | 0.750 | 0.413 | True
完成 | 0.600 | 0.500 | 1.000 | 0.109 | True
安全 | 0.030 | 0.040 | 0.750 | 0.256 | True
技术 prompt 输出： ['退款', '完成', '物流', '物流', '完成', '安全'] target_calls= 3 accepted= 5


## 六请求结果与 target 调用成本

In [4]:
speculative_results = []  # 收集六个推测解码结果
print("id | generated | tokens | target_calls | accepted | baseline_calls")  # 输出逐请求成本表头
for index, prompt in enumerate(prompts):  # 遍历六个业务 prompt
    tokens, calls, accepted, trace = speculative_decode(token_to_id[prompt[2]], 100 + index)  # 用与基线对应种子执行推测解码
    speculative_results.append((tokens, calls, accepted))  # 保存输出和调用成本
    print(f"{prompt[0]} | {[vocabulary[token] for token in tokens]} | {len(tokens)} | {calls} | {accepted} | {baseline_results[index][1]}")  # 展示生成结果与 target 调用对照
total_speculative_tokens = sum(len(row[0]) for row in speculative_results)  # 统计推测解码总 token 数
total_target_calls = sum(row[1] for row in speculative_results)  # 统计推测解码 target 调用数
total_accepted = sum(row[2] for row in speculative_results)  # 统计接受草稿 token 数
average_calls_per_token = total_target_calls / total_speculative_tokens  # 计算每生成 token 的昂贵调用
acceptance_rate = total_accepted / total_speculative_tokens  # 计算教学样本草稿接受率
print(f"总 token={total_speculative_tokens}，target calls={total_target_calls}，calls/token={average_calls_per_token:.3f}，acceptance={acceptance_rate:.1%}")  # 汇总推测收益

id | generated | tokens | target_calls | accepted | baseline_calls
SP-01 | ['物流', '<eos>'] | 2 | 1 | 2 | 3
SP-02 | ['完成', '退款', '<eos>'] | 3 | 2 | 2 | 4
SP-03 | ['退款', '完成', '物流', '物流', '完成', '安全'] | 6 | 3 | 5 | 5
SP-04 | ['完成', '<eos>'] | 2 | 1 | 2 | 2
SP-05 | ['<eos>'] | 1 | 1 | 1 | 1
SP-06 | ['<eos>'] | 1 | 1 | 1 | 1
总 token=15，target calls=9，calls/token=0.600，acceptance=86.7%


## 失败案例与修正：无条件接受 draft 会改变分布

从“排查”状态独立采样 2000 次下一 token。错误方案直接取 draft；修正方案执行一次 acceptance/residual sampling。比较经验频率到 target 分布的 L1 误差。

In [5]:
def one_step_corrected(previous, generator):  # 对单个 draft 提议执行严格修正采样
    target_distribution = target_probabilities[previous]  # 读取权威下一 token 分布
    draft_distribution = draft_probabilities[previous]  # 读取草稿下一 token 分布
    proposal, proposal_draw = categorical_sample(draft_distribution, generator)  # 从 q 采样一个提议
    threshold = min(1.0, float(target_distribution[proposal] / draft_distribution[proposal]))  # 计算接受概率
    if float(torch.rand((), generator=generator)) <= threshold:  # 按概率比判断是否接受
        return proposal  # 接受时直接返回 draft token
    residual = torch.clamp(target_distribution - draft_distribution, min=0.0)  # 计算被拒绝后的非负残差
    residual = residual / residual.sum()  # 归一化为修正分布
    correction, correction_draw = categorical_sample(residual, generator)  # 从残差分布采样替代 token
    return correction  # 返回保证 target 边缘分布的修正 token
naive_counts = torch.zeros(len(vocabulary))  # 初始化无条件 draft 经验计数
corrected_counts = torch.zeros(len(vocabulary))  # 初始化严格推测采样经验计数
naive_generator = torch.Generator().manual_seed(77)  # 创建错误方案固定随机流
corrected_generator = torch.Generator().manual_seed(77)  # 创建修正方案固定随机流
previous_id = token_to_id["排查"]  # 选择 draft 与 target 差异明显的状态
for sample_index in range(2000):  # 重复两千次估计边缘分布
    naive_token, naive_draw = categorical_sample(draft_probabilities[previous_id], naive_generator)  # 错误地无条件采用 draft
    corrected_token = one_step_corrected(previous_id, corrected_generator)  # 执行接受与残差修正
    naive_counts[naive_token] += 1  # 累加错误方案 token 次数
    corrected_counts[corrected_token] += 1  # 累加修正方案 token 次数
naive_frequency = naive_counts / naive_counts.sum()  # 归一化错误经验分布
corrected_frequency = corrected_counts / corrected_counts.sum()  # 归一化修正经验分布
naive_l1 = float((naive_frequency - target_probabilities[previous_id]).abs().sum())  # 计算无条件 draft 到 target 的 L1 误差
corrected_l1 = float((corrected_frequency - target_probabilities[previous_id]).abs().sum())  # 计算修正采样到 target 的 L1 误差
print("token | target | naive_draft | corrected")  # 输出经验分布对照表头
for token_id, token in enumerate(vocabulary):  # 遍历六个生成 token
    print(f"{token:5} | {target_probabilities[previous_id, token_id]:.3f} | {naive_frequency[token_id]:.3f} | {corrected_frequency[token_id]:.3f}")  # 展示每个 token 分布偏差
print(f"L1 error：naive={naive_l1:.4f}，corrected={corrected_l1:.4f}")  # 汇总分布正确性修复

token | target | naive_draft | corrected
退款    | 0.050 | 0.052 | 0.048
物流    | 0.050 | 0.060 | 0.052
排查    | 0.200 | 0.285 | 0.194
安全    | 0.050 | 0.043 | 0.052
完成    | 0.500 | 0.401 | 0.502
<eos> | 0.150 | 0.160 | 0.153
L1 error：naive=0.2120，corrected=0.0150


## 结果解读

接受轨迹展示了每个 token 的 p、q、概率比和随机门槛。高接受率时一个 target batch 验证多个 token，从而降低 calls/token。经验实验说明加速不能靠“永远信任小模型”，residual sampling 是保持 target 分布的关键。

## 生产边界

教学模型是 bigram 且 bonus forward 单独计费，真实 Transformer 会一次返回草稿块所有位置 logits，并复用 KV Cache。生产收益还受 draft 延迟、batch size、树状草稿、tokenizer 一致性和拒绝后的 cache 回滚影响。必须验证输出分布、吞吐、P50/P99 延迟及不同 prompt 长度接受率。

## 最小回归测试

In [6]:
assert len(prompts) >= 6  # 保证解码实验覆盖多个业务 prompt
assert all(len(tokens) >= 1 for tokens, calls, accepted in speculative_results)  # 保证每个请求真实生成 token
assert all(calls <= len(tokens) + 1 for tokens, calls, accepted in speculative_results)  # 保证 target 调用没有超过逐 token 基线量级
assert total_accepted > 0 and acceptance_rate > 0.3  # 保证 draft 在教学请求中有实际接受收益
assert average_calls_per_token < 1.0  # 保证批量验证降低每 token target 调用
assert corrected_l1 < naive_l1  # 保证 acceptance/residual 修正经验分布更接近 target
assert len(sample_trace) >= 1  # 保证保存了真实提议验证中间轨迹